In [1]:
# Install all required packages
!pip install datasets google-cloud-aiplatform pandas huggingface_hub ipywidgets

# Online-Mind2Web Fine-tuning with Vertex AI

This notebook demonstrates how to:
1. Load data from the gated `osunlp/Online-Mind2Web` dataset on Hugging Face.
2. Prepare the data for fine-tuning an LLM on Google Cloud Vertex AI.
3. **Integrate with `ai-qa-bot`**: Use the agent CLI to perform tasks and record actions.
4. Compare the agent's performance against the gold standard reference steps.

## 1. Setup and Authentication

First, authenticate with Hugging Face and Google Cloud.

In [2]:
from huggingface_hub import notebook_login

# You need to accept the conditions for osunlp/Online-Mind2Web on Hugging Face first.
# After that, run this cell and enter your Hugging Face Token.
notebook_login()

In [3]:
import os
from google.cloud import aiplatform

PROJECT_ID = "your-project-id"  # @param {type:"string"}
REGION = "us-central1"          # @param {type:"string"}

aiplatform.init(project=PROJECT_ID, location=REGION)

/home/daniel/projects/qa-tester/training/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/daniel/projects/qa-tester/training/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/home/daniel/projects/qa-tester/trai

## 2. Load Dataset

We use the `datasets` library to pull a record from the `osunlp/Online-Mind2Web` dataset.

In [4]:
from datasets import load_dataset

# Load the dataset in streaming mode to quickly pull one record
dataset = load_dataset("osunlp/Online-Mind2Web", split="test", streaming=True)
record = next(iter(dataset))

print("Task ID:", record.get('task_id'))
print("Website:", record.get('website'))
print("Task Description:", record.get('confirmed_task'))

# import json
# with open("sample_record.json", "r") as f:
#     record = json.load(f)
# print("Task ID:", record.get('task_id'))
# print("Website:", record.get('website'))
# print("Task Description:", record.get('confirmed_task'))

Task ID: b7258ee05d75e6c50673a59914db412e_110325
Website: https://www.traderjoes.com/
Task Description: Find the store location and hours of the closest Trader Joe's to zip code 90028 and set it as my home store.


## 3. Prepare for Vertex AI Fine-tuning

Vertex AI Gemini fine-tuning expects a JSONL format where each line represents a full conversation structure.

In [5]:
import json

def format_to_gemini_jsonl(record):
    """
    Formats a dataset record into Vertex AI Gemini fine-tuning JSONL format.
    """
    system_instruction = "You are a web agent that performs tasks on websites. Provide the action to take."
    user_prompt = f"Website: {record['website']}\nTask: {record['confirmed_task']}"
    
    # For finetuning, the gold standard would be the actual steps taken.
    gold_standard = f"Action: Click on the button related to {record['confirmed_task']}"
    
    entry = {
        "systemInstruction": {
            "role": "system",
            "parts": [{"text": system_instruction}]
        },
        "contents": [
            {
                "role": "user",
                "parts": [{"text": user_prompt}]
            },
            {
                "role": "model",
                "parts": [{"text": gold_standard}]
            }
        ]
    }
    return entry

training_sample = format_to_gemini_jsonl(record)
print("Formatted Sample for Vertex AI:")
print(json.dumps(training_sample, indent=2))

Formatted Sample for Vertex AI:
{
  "systemInstruction": {
    "role": "system",
    "parts": [
      {
        "text": "You are a web agent that performs tasks on websites. Provide the action to take."
      }
    ]
  },
  "contents": [
    {
      "role": "user",
      "parts": [
        {
          "text": "Website: https://www.traderjoes.com/\nTask: Find the store location and hours of the closest Trader Joe's to zip code 90028 and set it as my home store."
        }
      ]
    },
    {
      "role": "model",
      "parts": [
        {
          "text": "Action: Click on the button related to Find the store location and hours of the closest Trader Joe's to zip code 90028 and set it as my home store."
        }
      ]
    }
  ]
}


## 4. Run Test with ai-qa-bot CLI

We use the `ai-qa-bot` agent to execute the task on the live website. We stream the output to see logs in real-time.

In [7]:
import subprocess
import os
import sys

def run_agent_test(record):
    # Path to the ai-qa-bot directory
    agent_dir = "."
    goal = record['confirmed_task']
    url = record['website']
    task_id = record['task_id']
    output_file = f"../training/results/{task_id}.json"
    
    os.makedirs("../training/results", exist_ok=True)
    
    # CLI Command: npm run cli -- record "<Goal>" "<StartUrl>" "<OutputFile>"
    cmd = [
        "npm", "run", "cli", "--", 
        "record", goal, url, output_file,
        "--no-artifacts"
    ]
    
    print(f"--- Starting Agent for Task: {task_id} ---")
    print(f"Goal: {goal}")
    print(f"URL: {url}")
    print("\n--- Agent Output (Streaming) ---\n")
    
    try:
        # Using Popen to stream output
        process = subprocess.Popen(
            cmd, 
            cwd=agent_dir, 
            stdout=subprocess.PIPE, 
            stderr=subprocess.STDOUT, 
            text=True,
            bufsize=1
        )
        
        # Stream stdout line by line
        for line in iter(process.stdout.readline, ''):
            print(line, end='', flush=True)
            
        process.stdout.close()
        return_code = process.wait()
        
        if return_code == 0:
            print("\n--- Agent Success ---")
            return output_file
        else:
            print(f"\n--- Agent Failed with return code {return_code} ---")
            return None
            
    except Exception as e:
        print(f"\n--- Error running agent: {e} ---")
        return None

result_path = run_agent_test(record)
# print("Uncomment the line above to run the agent test with streaming logs.")

--- Starting Agent for Task: b7258ee05d75e6c50673a59914db412e_110325 ---
Goal: Find the store location and hours of the closest Trader Joe's to zip code 90028 and set it as my home store.
URL: https://www.traderjoes.com/

--- Agent Output (Streaming) ---


> web-testing-agent@1.0.0 cli
> ts-node src/cli.ts record Find the store location and hours of the closest Trader Joe's to zip code 90028 and set it as my home store. https://www.traderjoes.com/ ../training/results/b7258ee05d75e6c50673a59914db412e_110325.json --no-artifacts

[dotenv@17.3.1] injecting env (1) from .env -- tip: 🛡️ auth for agents: https://vestauth.com
[Browser] New page opened: about:blank
[CLI] Model: gemini-3.1-flash-lite-preview (Vision support: true)
[CLI] Starting Record Mode...
[Browser] Executing action: navigate
[Browser] Active page set to: about:blank (TargetID: 5EEC94E9850B682E03B9C1C533C83709)
[Browser] Active page set to: about:blank (TargetID: 5EEC94E9850B682E03B9C1C533C83709)
[Agent] Starting test execut

## 5. Compare against Gold Standard

We compare the agent's actual steps with the human reference steps.

In [8]:
def compare_performance(result_path, record):
    if not result_path or not os.path.exists(result_path):
        print("Result file not found.")
        return
        
    with open(result_path, 'r') as f:
        data = json.load(f)
    
    print(data.get('steps'))
    agent_steps = len(data.get('steps', []))
    human_steps = record.get('reference_length')
    
    print(f"Task: {record['confirmed_task']}")
    print(f"Agent Steps: {agent_steps}")
    print(f"Human Reference Steps: {human_steps}")
    
    diff = agent_steps - human_steps
    if diff <= 0:
        print("Performance: Optimal (Matched or beat human steps!)")
    else:
        print(f"Performance: Sub-optimal ({diff} more steps than human)")

print(f"result_path {result_path}")
compare_performance(result_path, record)

result_path None
Result file not found.


## 6. Next Steps: Batch Processing
